In [1]:
import logging
from pathlib import Path

import arcpy
from tqdm.notebook import tqdm

from arcpy_parquet import parquet_to_feature_class

In [2]:
data_dir = Path(r'C:\projects\foursquare-data-loading\data')
pqt_pth = data_dir / r'raw\foursquare_data_packs\parquet'
out_dir = data_dir / r'processed\foursquare_data_packs'

logging.basicConfig(level="DEBUG")

In [3]:
delivery_year = 2025
delivery_month = 1

# create partitioned path to where data will be stored
out_dir = out_dir / f'delivery_year={delivery_year}' / f'delivery_month={delivery_month}'

out_dir

WindowsPath('C:/projects/foursquare-data-loading/data/processed/foursquare_data_packs/delivery_year=2025/delivery_month=1')

In [5]:
# create parquet path to specific month - ensures delivery_* columns are not included in built data
pqt_date_pth = pqt_pth / f'delivery_year={delivery_year}' / f'delivery_month={delivery_month}'

if not pqt_date_pth.exists():
    raise FileNotFoundError(f"""The dataset for the specified year and month does not appear to exist, "{pqt_date_pth}\"""")
else:
    logging.info(f"""Using parquet dataset located at "{pqt_date_pth}\"""")

# use same schema file for all builds
schema_pth = pqt_pth.parent / 'schema'

assert schema_pth.exists()

schm_lst = list(schema_pth.glob('*.csv'))
assert len(schm_lst)

schema_pth = schm_lst[0]

schema_pth

INFO:root:Using parquet dataset located at "C:\projects\foursquare-data-loading\data\raw\foursquare_data_packs\parquet\delivery_year=2025\delivery_month=1"


WindowsPath('C:/projects/foursquare-data-loading/data/raw/foursquare_data_packs/schema/part-00000-47106081-e2b4-49ee-b8f7-fca8a4608d9a-c000.csv')

In [ ]:
pth_set = set(pth.parent for pth in pqt_pth.rglob('*.parquet'))

for pth in tqdm(pth_set):

    # get the part of the path defining the country
    cntry_prt = [prt for prt in pth.parts if prt.startswith('country')][0]
    
    # location to save the exported country data
    cntry_dir = out_dir / cntry_prt
    
    # make sure the directory exists
    if not cntry_dir.exists():
        cntry_dir.mkdir(parents=True)
    
    # create path to feature class
    fc_pth = cntry_dir / 'foursquare.gdb' / 'places'
    
    # create the file geodatabase to hydrate
    if arcpy.Exists(str(fc_pth.parent)):
        arcpy.management.Delete(str(fc_pth.parent))
        
    with arcpy.EnvManager(overwriteOutput=True):
        _ = arcpy.management.CreateFileGDB(str(cntry_dir), 'foursquare.gdb')
    
    # convert the data to a feature class
    parquet_to_feature_class(
        parquet_path=pqt_date_pth, 
        output_feature_class=fc_pth, 
        schema_file=schema_pth, 
        parquet_partitions=[cntry_prt], 
        geometry_type='COORDINATES',
        geometry_column=('longitude', 'latitude'),
        build_spatial_index=True, compact=True
    )
    
    logging.info(f'Successfully created {fc_pth}')